<!-- ![Alt Text](https://raw.githubusercontent.com/msfasha/307304-Data-Mining/main/images/header.png) -->

<div style="display: flex; justify-content: flex-start; align-items: center;">
   <a href="https://colab.research.google.com/github/msfasha/307307-BI-Methods-Generative-AI/blob/main/20252/Module 2 - Introduction to NNs and Word Embeddings/2-Introduction to Neural Networks and Word Embeddings-Python.ipynb" target="_parent"><img
   src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
</div>

## **The Perceptron**

<div style="text-align: center;">
  <img src="https://raw.githubusercontent.com/msfasha/307307-BI-Methods-Generative-AI/main/images/perceptron.png" alt="Simple Perceptron" width="500"/>
</div>

### Implement the Perceptron using scikit-learn library

In [23]:
from sklearn.linear_model import Perceptron
import numpy as np

# Training data for AND gate
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 0, 0, 1])

# Initialize and train Perceptron
model = Perceptron(max_iter=100, eta0=0.1, random_state=42)
model.fit(X, y)

# Results
print("Weights:", model.coef_)
print("Bias:", model.intercept_)
print("Predictions:", model.predict(X))

Weights: [[0.2 0.2]]
Bias: [-0.2]
Predictions: [0 0 0 1]


Note: In the scikit-learn Perceptron, the step function (also called the activation function) is a hard threshold function, and it's built-in.<br>

```Python
prediction = 1 if output >= 0 else 0

---

## **The Mulit-Layer Perceptron - MLP**

<div style="text-align: center;">
    <img src="https://raw.githubusercontent.com/msfasha/307307-BI-Methods-Generative-AI/main/images/mlp.png" alt="Multi Layer Perceptron" width="600"/>
</div>

### Solving the XOR Problem using a Neural Network

This code demonstrates how to build and train a simple neural network from scratch using NumPy to learn the XOR logic gate.

In [24]:
from sklearn.neural_network import MLPClassifier
import numpy as np

# XOR input and output
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])

# Define MLP with 1 hidden layer of 2 neurons (minimal config for XOR)
mlp = MLPClassifier(hidden_layer_sizes=(2,), activation='tanh',
                    solver='adam', learning_rate_init=0.01,
                    max_iter=10000, random_state=42)

# Train the model
mlp.fit(X, y)

# Make predictions
predictions = mlp.predict(X)

print("Predictions:\n", predictions)
print("\nWeights (input to hidden):\n", "[ w11 , w12 ]\n[ w21 , w22 ]\n", mlp.coefs_[0])
print("\nBias hidden:\n", mlp.intercepts_[0])
print("\nWeights (hidden to output):\n", mlp.coefs_[1])
print("\nBias output:\n", mlp.intercepts_[1])


Predictions:
 [0 1 1 0]

Weights (input to hidden):
 [ w11 , w12 ]
[ w21 , w22 ]
 [[ 2.7144501   3.27401218]
 [-2.73418453 -3.17014048]]

Bias hidden:
 [ 1.21994174 -1.63451199]

Weights (hidden to output):
 [[-4.37775211]
 [ 4.46553876]]

Bias output:
 [3.61855675]


---

## **Working with Real Embeddings - The Gensim Library**

For this example, we will use the `word2vec-google-news-300` model. <br>
The `word2vec-google-news-300` model is a **pre-trained Word2Vec model** created by Google.  
It has the following characteristics:

- **Training data**: Trained on approximately **100 billion words** from the **Google News** dataset.
- **Vector size**: Each word is represented as a **300-dimensional** vector.
- **Vocabulary size**: It contains **about 3 million unique words and phrases**.
- **Training method**: It uses the **skip-gram** Word2Vec architecture to predict context words given a target word.
- It's a little **heavy**, about 1.5 GB, so it's good to know the best way to handle it.

This model captures a wide range of **semantic** and **syntactic** relationships between words.  
Because it is trained on a large and diverse corpus, it is widely used for many natural language processing (NLP) tasks where high-quality word embeddings are required.

First, we need to install gensim library

In [25]:
%pip install gensim

#### Load English Word Embeddings

In [26]:
import gensim.downloader as api

# Load the pre-trained Word2Vec model
word2vec_model = api.load("word2vec-google-news-300")

#### Compute similarity between two words

In [27]:
similarity_score = word2vec_model.similarity('coffee', 'tea')
print(f"\nSimilarity between 'coffee' and 'tea': {similarity_score:.4f}")


Similarity between 'coffee' and 'tea': 0.5635


#### Find Similar Words

In [28]:
# Find words similar to 'computer'
similar_words = word2vec_model.most_similar('computer', topn=5)

print("Words similar to 'computer':")
for word, similarity in similar_words:
    print(f"{word}: {similarity:.4f}")

Words similar to 'computer':
computers: 0.7979
laptop: 0.6640
laptop_computer: 0.6549
Computer: 0.6473
com_puter: 0.6082


#### Example of a word analogy: king - man + woman = ?

In [29]:
analogy_result = word2vec_model.most_similar(positive=['woman', 'king'], negative=['man'], topn=3)
print("\nResult of analogy (king - man + woman):")
for word, similarity in analogy_result:
    print(f"{word}: {similarity:.4f}")


Result of analogy (king - man + woman):
queen: 0.7118
monarch: 0.6190
princess: 0.5902


#### Find odd one out

In [30]:
odd_one_out = word2vec_model.doesnt_match(["breakfast", "lunch", "dinner", "car"])
print("\nOdd one out in ['breakfast', 'lunch', 'dinner', 'car']:", odd_one_out)


Odd one out in ['breakfast', 'lunch', 'dinner', 'car']: car


---

### **Sentence Embeddings Example**

#### Semantic Similarity: Ranking Answers to a Question using Sentence Embeddings

Word2Vec gives each **word** a vector. But for comparing full sentences or phrases, a better approach is **sentence embeddings** — models that encode an *entire sentence* into a single vector that captures its overall meaning.

We will use the [`sentence-transformers`](https://www.sbert.net/) library with the `all-MiniLM-L6-v2` model:
- Lightweight (~80 MB) but highly effective
- Specifically trained to produce embeddings where **semantically similar sentences are close** in vector space
- Encodes full sentences in one shot — no need to average word vectors

**The Idea:**  
Encode the question and each candidate answer as sentence vectors, then rank the answers by their **cosine similarity** to the question.

| Score ≈ 1.0 | Score ≈ 0.0 |
|---|---|
| Very semantically related | Completely unrelated |

In the example below, the question is *"What is a good pet for children?"* Our 10 candidate answers include semantically related phrases (even full sentences like *"Dogs make loyal companions"*) and completely unrelated ones. The model should rank related answers at the top purely from meaning — **no shared words required**.

In [31]:
%pip install -q sentence-transformers

##### Step 1 — Load the model, encode all sentences, rank by similarity

The code below does the following:

1. **Loads** the `all-MiniLM-L6-v2` sentence-transformer model — a compact neural network fine-tuned to map sentences into a 384-dimensional vector space where *meaning* determines proximity.
2. **Defines** the question and 10 candidate answers (6 related, 4 unrelated). Notice that the related answers share **no keywords** with the question — the model must rely purely on semantic understanding.
3. **Encodes** all 11 sentences in one batch call (`model.encode`). Each sentence becomes a single fixed-size vector regardless of its length.
4. **Computes cosine similarity** between the question vector and every answer vector — a value between 0 and 1 where higher means more similar.
5. **Ranks** the answers from most to least relevant and prints the table.

In [32]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load a lightweight but effective sentence embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# --- Question and 10 candidate answers (full sentences) ---
question = "What is a good pet for children?"

answers = [
    ("Dogs make loyal and playful companions for kids.",          "related"),
    ("A kitten can be a gentle and affectionate pet.",            "related"),
    ("Rabbits are calm animals that children enjoy caring for.",  "related"),
    ("A puppy will bring joy and energy to any family.",          "related"),
    ("Guinea pigs are easy to handle and great for young kids.",  "related"),
    ("Hamsters are small, low-maintenance pets.",                 "related"),
    ("The stock market closed higher on Tuesday.",                "unrelated"),
    ("Photosynthesis converts sunlight into glucose.",            "unrelated"),
    ("The Eiffel Tower is located in Paris, France.",             "unrelated"),
    ("Algebra involves solving equations with variables.",        "unrelated"),
]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encode the question and all answers

In [33]:
# place question and answer in a single list
sentences    = [question] + [a[0] for a in answers]

Print the list

In [34]:
sentences

['What is a good pet for children?',
 'Dogs make loyal and playful companions for kids.',
 'A kitten can be a gentle and affectionate pet.',
 'Rabbits are calm animals that children enjoy caring for.',
 'A puppy will bring joy and energy to any family.',
 'Guinea pigs are easy to handle and great for young kids.',
 'Hamsters are small, low-maintenance pets.',
 'The stock market closed higher on Tuesday.',
 'Photosynthesis converts sunlight into glucose.',
 'The Eiffel Tower is located in Paris, France.',
 'Algebra involves solving equations with variables.']

Create sentence embeddings for the question and the answers

In [35]:
embeddings   = model.encode(sentences)

q_embedding  = embeddings[0]
a_embeddings = embeddings[1:]

Compute cosine similarities

In [36]:
sims = cosine_similarity([q_embedding], a_embeddings)[0]
sims

array([ 0.6582606 ,  0.36188948,  0.5143815 ,  0.45709622,  0.4941338 ,
        0.43262008, -0.02417872,  0.00393954,  0.04021309,  0.01213969],
      dtype=float32)

Build and sort results

In [40]:
results = [(answers[i][0], answers[i][1], float(sims[i])) for i in range(len(answers))]
results.sort(key=lambda x: x[2], reverse=True)
results

[('Dogs make loyal and playful companions for kids.',
  'related',
  0.6582605838775635),
 ('Rabbits are calm animals that children enjoy caring for.',
  'related',
  0.5143815279006958),
 ('Guinea pigs are easy to handle and great for young kids.',
  'related',
  0.4941338002681732),
 ('A puppy will bring joy and energy to any family.',
  'related',
  0.4570962190628052),
 ('Hamsters are small, low-maintenance pets.', 'related', 0.4326200783252716),
 ('A kitten can be a gentle and affectionate pet.',
  'related',
  0.36188948154449463),
 ('The Eiffel Tower is located in Paris, France.',
  'unrelated',
  0.04021309316158295),
 ('Algebra involves solving equations with variables.',
  'unrelated',
  0.012139685451984406),
 ('Photosynthesis converts sunlight into glucose.',
  'unrelated',
  0.003939539194107056),
 ('The stock market closed higher on Tuesday.',
  'unrelated',
  -0.024178724735975266)]

--- Print ranked results ---

In [ ]:
print(f'Question: "{question}"\n')
print(f"{'Rank':<6} {'Similarity':>10}  {'Label':<12}  Answer")
print("-" * 80)
for rank, (answer, label, sim) in enumerate(results, 1):
    print(f"{rank:<6} {sim:>10.4f}  {label:<12}  {answer}")

Question: "What is a good pet for children?"

Rank   Similarity  Label         Answer
--------------------------------------------------------------------------------
1          0.6583  related       Dogs make loyal and playful companions for kids.
2          0.5144  related       Rabbits are calm animals that children enjoy caring for.
3          0.4941  related       Guinea pigs are easy to handle and great for young kids.
4          0.4571  related       A puppy will bring joy and energy to any family.
5          0.4326  related       Hamsters are small, low-maintenance pets.
6          0.3619  related       A kitten can be a gentle and affectionate pet.
7          0.0402  unrelated     The Eiffel Tower is located in Paris, France.
8          0.0121  unrelated     Algebra involves solving equations with variables.
9          0.0039  unrelated     Photosynthesis converts sunlight into glucose.
10        -0.0242  unrelated     The stock market closed higher on Tuesday.
